In [0]:
census_df = spark.sql("""SELECT HnF.GEO_ID, hnf.NAME, pop.S0101_C01_001E AS Population,
    pop.state AS stateFP, pop.county AS countyFP, pop.tract AS tractCE,
    HnF.S1101_C01_002E AS Avg_household_size,
    hnf.S1101_C01_010E AS Households_w_ppl_under_18_y,
    hnf.S1101_C01_011E AS Households_w_ppl_60_plus_y,
    hnf.S1101_C01_012E AS Households_w_ppl_65_plus_y,
    hnf.S1101_C01_013E AS Householder_living_alone,
    hnf.S1101_C01_014E AS Lonely_Householder_65_plus_y,
    cmmt.S0801_C01_002E AS Car_truck_van,
    cmmt.S0801_C01_003E AS Drive_alone,
    cmmt.S0801_C01_004E AS carpooled,
    cmmt.S0801_C01_009E AS Pub_transport,
    cmmt.S0801_C01_010E AS walked,
    cmmt.S0801_C01_011E AS bicycle_2_work,
    cmmt.S0801_C01_012E AS Taxi_ride_hail_motorbike_other,
    cmmt.S0801_C01_013E AS Worked_from_home,
    pov.S1701_C03_001E AS pct_below_poverty,
    gp.S1002_C01_030E AS household_w_grand_parents_children,
    work_status.S2303_C01_031E AS avg_weekly_hr_worked_per_worker,
    occp_char.S2501_C02_002E AS pct_occ_household_1_person,
    occp_char.S2501_C02_003E AS pct_occ_household_2_person,
    occp_char.S2501_C02_004E AS pct_occ_household_3_person,
    occp_char.S2501_C02_005E AS pct_occ_household_4_n_up,
    fin_char.S2503_C01_013E AS median_household_income,
    health_ins.S2701_C03_001E AS pct_health_insured,
    pbl_health_ins.S2704_C03_001E AS pct_pbl_health_insured
FROM ca_healthcare_fac_bronze.census_data_ingest_bronze.s1101 AS HnF       --Households and Families 
Left JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s0801 AS cmmt        -- Commuting Characteristics 
ON HnF.GEO_ID = cmmt.GEO_ID
LEFT JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s0101 AS pop
ON HnF.GEO_ID = pop.GEO_ID
Left JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s1701 AS pov         -- poverty_status in the past 12 months
ON HnF.GEO_ID = pov.GEO_ID
LEFT JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s1002 AS gp          --      GrandParents
ON HnF.GEO_ID = gp.GEO_ID
LEFT JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s2303 AS work_status   -- Work Status
ON HnF.GEO_ID = work_status.GEO_ID
LEFT JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s2501 AS occp_char      -- Occupancy Character
ON HnF.GEO_ID = occp_char.GEO_ID
LEFT JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s2503 AS fin_char      -- Financial Character
ON HnF.GEO_ID = fin_char.GEO_ID      
LEFT JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s2701 AS health_ins          -- Health Insurance Coverage
ON HnF.GEO_ID = health_ins.GEO_ID
LEFT JOIN ca_healthcare_fac_bronze.census_data_ingest_bronze.s2704 AS pbl_health_ins        -- Public Health Insurance Coverage
ON HnF.GEO_ID = pbl_health_ins.GEO_ID
ORDER BY HnF.GEO_ID ASC""")
display(census_df)

Bring in the MSSA fields based on the centroid latitudes and longitudes of census tracts

In [0]:
mssa_bronze_pddf = spark.read.table("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo").toPandas()
print(mssa_bronze_pddf.columns)

In [0]:
census_df.printSchema()

In [0]:
mssa_bronze_pddf.printSchema()


In [0]:
census_df[census_df['tractCE']== '302005'].show()

In [0]:
mssa_bronze_pddf.head()

Since MSSAs are consisted with multiple censustracts, do a straight forward table join on the GEOIDs